# Baseline：TF-IDF + 逻辑回归

**负责人：** 1 号　**阶段：** 1 数据准备

建立所有高级模型都要对比的基准（阶段 3i）。实现写在 `member1/baseline.py`，本 notebook 调用它；测试在 `member1/test_baseline.py`。默认的 `MODE = "team"` 只使用正式划分 `splits/folds_v1.csv`，划分或环境没准备好时只显示原因，不会自己生成替代划分。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 在 `text_norm` 上做 TF-IDF + 逻辑回归，TF-IDF 只在每个训练折内 fit
- [x] 输出 Accuracy、Precision、Recall、F1 和混淆矩阵
- [x] 画出每个类别权重最高的词
- [x] 每次实验先用 `new_run_id()` 生成 run_id，评分记录和预测文件都带上同一个 run_id
- [ ] 在正式划分上运行（`MODE = "team"`），结果写进中期报告
- [ ] 测试集预测概率（11/18 前；Stacking 和最终对比要用）
- [ ] 阶段 3iv（必做）：调参后重新评估，再和 6 个模型比较一次
- [ ] （选做）字符 n-gram、投票集成

## 产出

- `results/baseline.csv`；OOF 概率用 `save_preds(run_id, ...)` 存到 `DATA_ROOT/preds/`
- `figures/baseline_<run_id>_confusion.png`、`figures/baseline_<run_id>_top_words.png`
- 每次运行的完整输出（各折成绩、分类报告、OOF 预测）在 `member1/outputs/<run_id>/`，不进 Git

目前测试集推理延后，预测文件里的 test 数组为空，下游做集成前要先确认测试集预测已补齐。

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写
from member1.baseline import readiness, run                                 # 1 号的 baseline 实现

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
import pandas as pd
from IPython.display import Image, display

status = readiness()
display(status)
clean = load_clean()
display(clean.groupby(["label", "label_name"]).size().rename("samples").to_frame())

## Experiment configuration

`MODE="team"` 使用正式 Data Splits 和团队 Environment。

仅需调试时可手动设为 `MODE="provisional"`，临时 Results 只保存到 `member1/outputs/`，不会进入团队 Results 或 Prediction Cache。

每次运行都会生成新的 run_id。TF-IDF 在每个 Training Fold 内独立 fit；同一 group_id 不跨 Fold。

In [ ]:
MODE = "team"
output = None
if MODE == "team" and not status["ready_for_team_run"]:
    print("Team Run pending. Check input_errors and environment_mismatches above.")
else:
    output = run(mode=MODE)
    print(output)

In [ ]:
if output is not None:
    import json
    summary = json.loads((output / "summary.json").read_text())
    display(pd.DataFrame({"CV mean": summary["cv_mean"], "CV standard deviation": summary["cv_std_sample"]}))
    display(pd.read_csv(output / "classification_report.csv", index_col=0))
    display(Image(filename=str(output / "confusion_matrix.png")))
    display(Image(filename=str(output / "top_words.png")))

## Interpretation and reporting

- `results/baseline.csv` 的团队 Metrics 使用 pooled OOF Predictions；各 Fold 的 mean 和 sample standard deviation 另存于本次输出目录。
- Confusion Matrix 的 rows 为 True Labels，columns 为 Predicted Labels。
- Top Words 按各 Class 的正向 LR Coefficients 排列，不是 Word Frequency，也不能解释为 Causality。
- Fit time 为五个 Training Folds 的 TF-IDF + LR fit 总时间，不含 Plotting、Inference 或解释用 Refit。
- Methodology 初稿见 `docs/report_drafts/m1_baseline.md`；实验记录见 `docs/experiments/m1.md`。
- 最终单个 Notebook 合并时，需要将 `member1/baseline.py` 的函数一并内联，避免依赖遗漏。
